In [11]:
import pandas as pd


ruta_archivo = 'dataset.csv' 
df_musica = pd.read_csv(r"C:\Users\lenovo\Documents\ASemestre 4\Analisis\PROYECTO\dataset.csv")

def obtener_datos_cancion_local(nombre_cancion, df):
    resultado = df[df['track_name'].str.contains(nombre_cancion, case=False, na=False)]
    
    if resultado.empty:
        return {"error": f"No se encontró la canción '{nombre_cancion}' en el dataset."}
    
    cancion = resultado.iloc[0]
    
    # Añadimos loudness y acousticness al diccionario extraído
    datos = {
        'id': cancion.get('track_id', 'Sin ID'),
        'nombre': cancion.get('track_name', nombre_cancion),
        'artista': cancion.get('artists', 'Artista Desconocido'),
        'duracion_ms': cancion.get('duration_ms', 0),
        'popularidad': cancion.get('popularity', 0), 
        'danceability': cancion.get('danceability', 0.0),
        'energy': cancion.get('energy', 0.0),
        'tempo_bpm': cancion.get('tempo', 0.0),
        'valence': cancion.get('valence', 0.0),
        'loudness': cancion.get('loudness', 0.0),          
        'acousticness': cancion.get('acousticness', 0.0)   
    }
    return datos

En la parte anterior, organizamos todos los datos del dataset y las columnas que tenemos

In [2]:
import requests
from bs4 import BeautifulSoup

def scrape_billboard_top_10():
    url = "https://www.billboard.com/charts/hot-100/"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    canciones = []
    # Las clases de Billboard cambian a menudo, esta es la estructura general actual
    items = soup.find_all('div', class_='o-chart-results-list-row-container')
    
    for item in items[:10]: # Solo el top 10 para probar
        titulo = item.find('h3', id='title-of-a-story').text.strip()
        artista = item.find('span', class_='c-label').text.strip()
        canciones.append({'titulo': titulo, 'artista': artista})
        
    return pd.DataFrame(canciones)

df_billboard = scrape_billboard_top_10()

Ahora en esta parte de codigo ya realizamos el webscrapping de billboard
Procedemos a hacer la limpieza de datos de nuestro dataset.

In [12]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

def limpiar_datos(df):
    # 1. Eliminar duplicados
    df = df.drop_duplicates(subset=['id_cancion'])
    
    # 2. Manejo de valores nulos
    # Llenar nulos en variables numéricas con la mediana
    num_cols = ['danceability', 'energy', 'tempo_bpm', 'loudness', 'acousticness']
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    
    # Eliminar filas si falta el nombre o el artista
    df = df.dropna(subset=['nombre', 'artista'])
    
    # 3. Normalización de texto (para cruzar datos de Spotify y Billboard)
    df['nombre'] = df['nombre'].str.lower().str.strip()
    df['artista'] = df['artista'].str.lower().str.strip()
    
    # 4. Transformación de variables
    # Convertir duración de milisegundos a minutos
    if 'duracion_ms' in df.columns:
        df['duracion_minutos'] = df['duracion_ms'] / 60000
        df = df.drop(columns=['duracion_ms'])
    
    # 5. Escalar variables numéricas para el modelo predictivo
    scaler = MinMaxScaler()
    variables_a_escalar = ['danceability', 'energy', 'tempo_bpm', 'valence', 'duracion_minutos']
    df[variables_a_escalar] = scaler.fit_transform(df[variables_a_escalar])
    
    return df



In [4]:

# ZONA DE PRUEBAS

print(" PRUEBA DATASET LOCAL ")

# Busca una canción que sepas que es famosa para ver si está en tu archivo
cancion_prueba = "Positions"
mis_datos = obtener_datos_cancion_local(cancion_prueba, df_musica)

if "error" in mis_datos:
    print(mis_datos["error"])
else:
    print(f"Éxito! Datos de {mis_datos['nombre']} por {mis_datos['artista']}:")
    print(f"   - Popularidad: {mis_datos['popularidad']}/100")
    print(f"   - Bailable: {mis_datos['danceability']}")
    print(f"   - Energía: {mis_datos['energy']}")
    print(f"   - Tempo (BPM): {mis_datos['tempo_bpm']}")

# 2. Probamos Billboard
print("\n📈 --- PRUEBA BILLBOARD --- 📈")
try:
    df_top10 = scrape_billboard_top_10()
    print("¡Éxito! Aquí está el Top 5 actual:")
    print(df_top10.head(5))
except Exception as e:
    print("Error leyendo Billboard:", e)

 PRUEBA DATASET LOCAL 
Éxito! Datos de positions por Ariana Grande:
   - Popularidad: 85/100
   - Bailable: 0.737
   - Energía: 0.802
   - Tempo (BPM): 144.015

📈 --- PRUEBA BILLBOARD --- 📈
¡Éxito! Aquí está el Top 5 actual:
            titulo artista
0      Janice STFU       1
1   Ran To Atlanta       2
2  Whisper My Name       3
3          Shabang       4
4   Choosin' Texas       5


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings

# Ignorar advertencias menores de Pandas
warnings.filterwarnings('ignore')

class CerebroPredictivo:
    def __init__(self):
        
        self.modelo = RandomForestClassifier(
            n_estimators=200, 
            max_depth=15,
            class_weight='balanced', 
            random_state=42
        )
        
    def entrenar_con_dataframe(self, df):
        print("\nINICIANDO CEREBRO PREDICTIVO ")
        
       
        columnas_necesarias = ['tempo', 'danceability', 'energy', 'valence', 'loudness', 'acousticness', 'popularity']
        
        if not all(col in df.columns for col in columnas_necesarias):
            print("Error: Faltan columnas en el dataset. Verifica que tengas loudness y acousticness.")
            return False

        df_modelo = df.dropna(subset=columnas_necesarias).copy()
        
  
        umbral_hit = 60
        df_modelo['es_hit'] = (df_modelo['popularity'] >= umbral_hit).astype(int)
        
        # Variables independientes (X) y objetivo (y) actualizadas
        X = df_modelo[['tempo', 'danceability', 'energy', 'valence', 'loudness', 'acousticness']]
        y = df_modelo['es_hit']
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        self.modelo.fit(X_train, y_train)
        
        precision = accuracy_score(y_test, self.modelo.predict(X_test))
        print(f"Modelo entrenado con {len(df_modelo)} canciones.")
        print(f"Total de 'Hits' detectados en el entrenamiento: {y.sum()}")
        print(f"Precisión del modelo base: {precision * 100:.2f}%")
        return True

    def calcular_probabilidad_viral(self, tempo, danceability, energy, valence, loudness, acousticness):
        # Actualizamos para recibir y procesar las nuevas variables
        datos_demo = pd.DataFrame([[tempo, danceability, energy, valence, loudness, acousticness]], 
                                  columns=['tempo', 'danceability', 'energy', 'valence', 'loudness', 'acousticness'])
        probabilidades = self.modelo.predict_proba(datos_demo)
        return probabilidades[0][1]


#Zona de prueba
if __name__ == "__main__":
    
    
    mi_ia = CerebroPredictivo()
    entrenamiento_exitoso = mi_ia.entrenar_con_dataframe(df_musica)
    
    if entrenamiento_exitoso:
        print("\nINICIANDO MODO PRODUCTOR")
        print("Bienvenido al laboratorio. Ingresa las métricas acústicas de tu demo.")
        
        try:
            nombre_demo = input("➤ Nombre de tu pista/demo: ")
            
            print("\nIntroduce los valores numéricos:")
            demo_tempo = float(input(" - Tempo (BPM, ej. 120.0): "))
            demo_danceability = float(input(" - Danceability (0.0 a 1.0): "))
            demo_energy = float(input(" - Energía (0.0 a 1.0): "))
            demo_valence = float(input(" - Positividad / Valence (0.0 a 1.0): "))
            
        
            # Loudness suele ser un valor negativo en Spotify (ej. -5.4)
            demo_loudness = float(input(" - Loudness/Volumen (en dB, ej. -4.5): ")) 
            # Acousticness es baja en canciones producidas por computadora y alta en acústicas
            demo_acousticness = float(input(" - Acousticness (0.0 a 1.0, ej. 0.01 para pop, 0.8 para balada a guitarra): "))
            
            print(f"\nProcesando el ADN Acústico de '{nombre_demo}'...")
            
            probabilidad = mi_ia.calcular_probabilidad_viral(
                tempo=demo_tempo,
                danceability=demo_danceability,
                energy=demo_energy,
                valence=demo_valence,
                loudness=demo_loudness,
                acousticness=demo_acousticness
            )
            
            print("\nRESULTADO DE HIT PREDICTOR")
            print(f"Probabilidad matemática de ser un Éxito Viral: {probabilidad * 100:.2f}%")
            
            if probabilidad > 0.70:
                print("Veredicto:Tiene un ADN altamente compatible con los éxitos.")
            elif probabilidad > 0.45:
                print("Veredicto: Buen potencial, pero el terreno es competitivo.")
            else:
                print("Veredicto: Riesgo alto de pasar desapercibida. Sugiere reestructuración.")

        except ValueError:
            print("\n Error: Por favor ingresa solo números válidos. Usa punto (.) para decimales.")
            
   
    print("\nCONTRASTE EN TIEMPO REAL: BILLBOARD HOT 100")
    print("Compara tu demo con lo que está dominando el mercado HOY:")
    try:
        df_top10 = scrape_billboard_top_10()
        print(df_top10.head(5).to_string(index=False))
    except Exception as e:
        print("Error leyendo Billboard:", e)


INICIANDO CEREBRO PREDICTIVO 
Modelo entrenado con 114000 canciones.
Total de 'Hits' detectados en el entrenamiento: 14822
Precisión del modelo base: 77.30%

INICIANDO MODO PRODUCTOR
Bienvenido al laboratorio. Ingresa las métricas acústicas de tu demo.

Introduce los valores numéricos:

Procesando el ADN Acústico de 'dynamite'...

RESULTADO DE HIT PREDICTOR
Probabilidad matemática de ser un Éxito Viral: 47.13%
Veredicto: Buen potencial, pero el terreno es competitivo.

CONTRASTE EN TIEMPO REAL: BILLBOARD HOT 100
Compara tu demo con lo que está dominando el mercado HOY:
         titulo artista
    Janice STFU       1
 Ran To Atlanta       2
Whisper My Name       3
        Shabang       4
 Choosin' Texas       5
